In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 484.9/484.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 11.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
from google.colab import files
uploaded = files.upload()

# Chargement du dataset
df = pd.read_csv("dataset_phishing.csv")

Saving dataset_phishing.csv to dataset_phishing.csv


In [ ]:
#Supposons que les colonnes sont ['url', 'label']
df = df[['url', 'status']]  # Adapte selon les vraies colonnes
df = df.dropna()  # Supprimer les lignes vides

In [ ]:
# 📌 1. Renommer la colonne "status" en "label"
df = df.rename(columns={"status": "label"})

In [ ]:
# 📌 2. Vérifier les valeurs uniques de "label"
print("Valeurs uniques de label:", df["label"].unique())

Valeurs uniques de label: ['legitimate' 'phishing']


In [ ]:
# 📌 2. Convertir "legitimate" en 0 et "phishing" en 1
df["label"] = df["label"].replace({"legitimate": 0, "phishing": 1})

<ipython-input-7-4abf09da3f12>:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["label"] = df["label"].replace({"legitimate": 0, "phishing": 1})


In [ ]:
# 📌 3. Convertir en int
df["label"] = df["label"].astype(int)

# 📌 4. Vérification
print(df.dtypes)  # label doit être int
print(df["label"].unique())  # Doit afficher [0, 1]

url      object
label     int64
dtype: object
[0 1]


In [ ]:
# 📌 3. Séparer les données en train et test
X_train, X_test, y_train, y_test = train_test_split(df['url'], df['label'], test_size=0.2, random_state=42)


In [ ]:
# 📌 4. Charger le tokenizer RoBERTa
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

In [ ]:
# 📌 5. Fonction de tokenisation
def tokenize_function(examples):
    return tokenizer(examples["url"], padding="max_length", truncation=True, max_length=256)

In [ ]:
# 📌 6. Convertir en Dataset Hugging Face
train_data = Dataset.from_pandas(pd.DataFrame({'url': X_train, 'label': y_train}))
test_data = Dataset.from_pandas(pd.DataFrame({'url': X_test, 'label': y_test}))

In [ ]:
# Appliquer la tokenisation
train_data = train_data.map(tokenize_function, batched=True)
test_data = test_data.map(tokenize_function, batched=True)

Map:   0%|          | 0/9144 [00:00<?, ? examples/s]

Map:   0%|          | 0/2286 [00:00<?, ? examples/s]

In [ ]:
# 📌 7. Convertir 'label' en entier
train_data = train_data.rename_column("label", "labels")
test_data = test_data.rename_column("label", "labels")

In [ ]:
# 📌 8. Charger le modèle RoBERTa pour la classification binaire
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# 📌 9. Définir les arguments d'entraînement
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    report_to="none"
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# 📌 10. Fonction de métriques
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1).numpy()
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


In [ ]:
# 📌 11. Initialiser le Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

<ipython-input-18-cdc61030dabe>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# 📌 12. Entraîner le modèle
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.306800,0.195106,0.940507,0.964453,0.913198,0.938126
2,0.165900,0.195552,0.951881,0.956951,0.945084,0.950980
3,0.107300,0.209864,0.956255,0.973321,0.937112,0.954874


TrainOutput(global_step=1716, training_loss=0.17946125688530626, metrics={'train_runtime': 1329.9253, 'train_samples_per_second': 20.627, 'train_steps_per_second': 1.29, 'total_flos': 3608831235317760.0, 'train_loss': 0.17946125688530626, 'epoch': 3.0})

In [ ]:
# 📌 13. Évaluer le modèle
results = trainer.evaluate()
print("Résultats :", results)

Résultats : {'eval_loss': 0.19510561227798462, 'eval_accuracy': 0.9405074365704287, 'eval_precision': 0.9644527595884004, 'eval_recall': 0.9131975199291408, 'eval_f1': 0.9381255686988171, 'eval_runtime': 28.5264, 'eval_samples_per_second': 80.136, 'eval_steps_per_second': 1.262, 'epoch': 3.0}
